# KVTC — KV-Cache Tensor Compression Demo

**Compress LLM KV caches 6-9x with negligible quality loss.**

This notebook demonstrates KVTC's three-stage compression pipeline:
1. **PCA decorrelation** — project KV vectors into principal components
2. **DP-optimal quantization** — allocate 0-16 bits per component via dynamic programming
3. **Entropy coding** — lossless compression on the quantized stream

[![GitHub](https://img.shields.io/badge/GitHub-OnlyTerp/kvtc-blue?logo=github)](https://github.com/OnlyTerp/kvtc)
[![arXiv](https://img.shields.io/badge/arXiv-2511.01815-b31b1b.svg)](https://arxiv.org/abs/2511.01815)

## 1. Setup

In [ ]:
# Install KVTC
!pip install -q git+https://github.com/OnlyTerp/kvtc.git
!pip install -q torch transformers datasets matplotlib

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import time

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
else:
    print('No GPU detected - running on CPU (demo still works, just slower)')
print(f'PyTorch: {torch.__version__}, Device: {device}')

## 2. Run Unit Tests

KVTC has 38 unit tests that verify the entire pipeline.

In [ ]:
!python -m pytest src/test_kvtc.py -v --tb=short 2>&1 | tail -50

## 3. Understanding the Pipeline

Walk through each stage of KVTC compression with synthetic data.

In [ ]:
from src.pca import compute_pca_basis, pca_transform, pca_inverse
from src.quantize import dp_allocate_bits, quantize_uniform, dequantize_uniform

torch.manual_seed(42)
seq_len, head_dim = 512, 128

# Generate correlated data (like real KV cache)
eigenvalues = torch.exp(-torch.arange(head_dim, dtype=torch.float32) * 0.1)
random_rotation = torch.linalg.qr(torch.randn(head_dim, head_dim))[0]
raw_data = torch.randn(seq_len, head_dim) * eigenvalues.sqrt() @ random_rotation.T
print(f'Synthetic KV cache: {raw_data.shape} ({raw_data.numel() * 2 / 1024:.1f} KB in FP16)')

### Stage 1: PCA Decorrelation

PCA finds the principal directions of variance. After transformation, dimensions are ordered by importance.

In [ ]:
mean = raw_data.mean(dim=0)
centered = raw_data - mean
cov = (centered.T @ centered) / (seq_len - 1)
eigenvals, eigenvecs = torch.linalg.eigh(cov)
idx = eigenvals.argsort(descending=True)
eigenvals, eigenvecs = eigenvals[idx], eigenvecs[:, idx]
pca_coeffs = centered @ eigenvecs

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
cumulative = eigenvals.cumsum(0) / eigenvals.sum()
axes[0].bar(range(50), eigenvals[:50].numpy(), color='#1a73e8', alpha=0.7)
ax2 = axes[0].twinx()
ax2.plot(range(50), cumulative[:50].numpy(), 'r-', lw=2, label='Cumulative')
ax2.axhline(y=0.99, color='green', ls='--', alpha=0.5, label='99% variance')
axes[0].set_xlabel('PCA Component'); axes[0].set_ylabel('Eigenvalue', color='#1a73e8')
ax2.set_ylabel('Cumulative Variance', color='red'); ax2.legend()
axes[0].set_title('Eigenvalue Spectrum')
after_corr = np.corrcoef(pca_coeffs[:, :10].numpy().T)
im = axes[1].imshow(np.abs(after_corr), vmin=0, vmax=1, cmap='Blues')
axes[1].set_title('Correlation after PCA (should be diagonal)')
plt.colorbar(im, ax=axes[1], label='|Correlation|')
plt.tight_layout(); plt.show()
n99 = (cumulative >= 0.99).nonzero()[0][0].item()
print(f'99% variance in first {n99}/{head_dim} components. Rest can be pruned.')

### Stage 2: DP-Optimal Bit Allocation

In [ ]:
budgets = [128, 256, 384, 512]
labels = ['1 b/d', '2 b/d', '3 b/d', '4 b/d']
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharey=True)
for i, (b, l) in enumerate(zip(budgets, labels)):
    bits = dp_allocate_bits(eigenvals, b)
    colors = ['#1a73e8' if x > 0 else '#e0e0e0' for x in bits[:50]]
    axes[i].bar(range(50), bits[:50].numpy(), color=colors)
    axes[i].set_xlabel('Component')
    if i == 0: axes[i].set_ylabel('Bits')
    n_active = (bits > 0).sum().item()
    axes[i].set_title(f'{l} (budget={b})\n{n_active} active, {head_dim-n_active} pruned', fontsize=9)
plt.suptitle('DP-Optimal Bit Allocation', fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

### Stage 3: Full Compression Roundtrip

In [ ]:
results = []
for budget in [64, 128, 192, 256, 384, 512, 768, 1024]:
    bits = dp_allocate_bits(eigenvals, budget)
    recon_pca = torch.zeros_like(pca_coeffs)
    for j in range(head_dim):
        if bits[j] > 0:
            col = pca_coeffs[:, j]
            ix, s, z = quantize_uniform(col, int(bits[j].item()))
            recon_pca[:, j] = dequantize_uniform(ix, s, z, int(bits[j].item()))
    recon = recon_pca @ eigenvecs.T + mean
    cos = torch.nn.functional.cosine_similarity(raw_data, recon, dim=-1).mean().item()
    bpd = budget / head_dim; comp = 16 / bpd
    results.append({'bpd': bpd, 'comp': comp, 'cos': cos})
    print(f'Budget {budget:4d} ({bpd:.1f} b/d) | {comp:5.1f}x | cos={cos:.6f}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot([r['comp'] for r in results], [r['cos'] for r in results], 'o-', color='#1a73e8', lw=2, ms=8)
for r in results:
    ax.annotate(f"{r['bpd']:.1f}b", (r['comp'], r['cos']), textcoords='offset points', xytext=(8,5), fontsize=8, color='#666')
ax.axhline(y=0.996, color='green', ls='--', alpha=0.5, label='KVTC K2V4 target (0.996)')
ax.set_xlabel('Compression Ratio', fontsize=12, fontweight='bold')
ax.set_ylabel('Cosine Similarity', fontsize=12, fontweight='bold')
ax.set_title('KVTC: Quality vs Compression', fontsize=13, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

## 4. What's Next?

This demo shows the core KVTC algorithm on synthetic data. For real-world usage:

1. **Calibrate on your model**: Run `src/calibrate.py` with a small text dataset (~10 samples)
2. **Use GPU-accelerated pipeline**: `KVTCCompressorFast` uses PyTorch GPU ops for 10x speedup
3. **Try different configs**: K1V3 (8.8x, good), K2V4 (6.1x, excellent), K4V6 (3.4x, near-lossless)

**[GitHub repo](https://github.com/OnlyTerp/kvtc) | [Paper](https://arxiv.org/abs/2511.01815) | [Benchmarks](https://github.com/OnlyTerp/kvtc/blob/master/BENCHMARKS.md)**

If this helps your research, give it a star!